# HomeReach Melbourne — 06: Match transport stops to LGAs

**Goal:** produce one record per `(feed_id, stop_id)` for the selected feeds' stop/platform locations, and attach an LGA only for a unique strict-interior match. Keep outside, exact-boundary, ambiguous and invalid-coordinate outcomes visible.

**Inputs:** the exact audited `gtfs.zip`, notebook 05's local audit summary, and your geography pipeline's validated 31-LGA GeoPackage/crosswalk/report. **No downloads, registry updates, rental changes or service-frequency calculations.**

Use the existing **homereach** kernel. Run original code cells top-to-bottom with **Ctrl+Enter** or **Run All**, not Code Runner. The startup cell prints before imports. We read only the three `stops.txt` tables, not the 11.8 million stop-time records again.

**Important:** these are location records, not counts of railway stations, physical facilities, active services, accessible journeys or transport quality. Feed 2 is an operational feed and includes replacement-bus information. The planned later service date is `2026-10-13`; this notebook does **not** filter locations by that date.

## 1. Imports and project paths

A point is built from longitude (x) and latitude (y). The LGA shapes are polygons. A spatial join will compare where points lie, not whether place names look similar. Hashes protect the earlier outputs from accidental changes.

In [1]:
print("Starting stop-to-LGA imports...", flush=True)
from pathlib import Path
from zipfile import ZipFile
from tempfile import SpooledTemporaryFile, TemporaryDirectory
from datetime import datetime, timezone
import hashlib
import io
import json
import shutil
import sys

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import pyproj
import shapely
from IPython.display import display
from homereach.geography.lga import load_geo_snapshot, load_aliases, build_crosswalk, CROSSWALK_COLUMNS

# Keep this notebook offline, including any optional PROJ grid retrieval.
pyproj.network.set_network_enabled(False)

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def find_root(start):
    for folder in [start.resolve(), *start.resolve().parents]:
        if (folder / "pyproject.toml").is_file() and (folder / "data").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook within the HomeReach project.")

ROOT = find_root(Path.cwd())
ZIP_PATH = ROOT / "data/raw/transport/gtfs.zip"
GTFS_AUDIT = ROOT / "data/interim/gtfs_audit/audit_summary.json"
GEO_SPEC_PATH = ROOT / "config/lga_snapshot.json"
ALIASES_PATH = ROOT / "config/lga_name_aliases_2025.csv"
GPKG_PATH = ROOT / "data/processed/geography/metro_lga_2025.gpkg"
CROSSWALK_PATH = ROOT / "data/interim/geography/lga_crosswalk_2025.csv"
GEO_REPORT_PATH = ROOT / "data/interim/geography/geography_pipeline_report.json"
OUT_DIR = ROOT / "data/interim/gtfs_spatial"
REPORT_PATH = ROOT / "docs/gtfs_stop_spatial_join.md"
EXPECTED_GTFS_SHA256 = "c88af3abcaf0872e45196dcc1d5af9bc20a82c54a00a9dfef46c809ec2e34814"
SELECTED_FEEDS = ("2", "3", "4")
FEED_LABELS = {"2": "Metropolitan train operational feed", "3": "Metropolitan tram", "4": "Myki bus: metropolitan + regional town"}
EXPECTED_LOCATIONS = {"2": 2859, "3": 1623, "4": 22633}
EXPECTED_PLATFORMS = {"2": 775, "3": 1623, "4": 22633}
POINT_EPSG = 4326
MATCH_EPSG = 7855  # GDA2020 / MGA zone 55; metres, covering the study area.
BOUNDARY_REVIEW_METRES = 10.0  # Diagnostic screen, not an accuracy guarantee or reassignment rule.

pd.set_option("display.max_rows", 40)
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 65)
print("Python:", sys.executable)
print("pandas:", pd.__version__, "| GeoPandas:", gpd.__version__, "| Shapely:", shapely.__version__)
print("Project root:", ROOT)

Starting stop-to-LGA imports...
Python: c:\Users\PhamDang\anaconda3\envs\homereach\python.exe
pandas: 3.0.3 | GeoPandas: 1.2.0 | Shapely: 2.1.2
Project root: D:\My project\homereach_melbourne


c:\Users\PhamDang\anaconda3\envs\homereach\Lib\site-packages\pyogrio\core.py:38: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


## 2. Verify the two upstream milestones

We verify the ZIP against notebook 05 and the boundary files against the geography build report. File fingerprints depend on the actual local export, so they are read from **your** report, not copied from another machine's GeoPackage. A mismatch stops here; do not edit a hash to bypass it.

Windows report paths may use backslashes. We normalise path separators only when reading the manifest keys. Nothing is renamed or written.

In [2]:
required_paths = [ZIP_PATH, GTFS_AUDIT, GEO_SPEC_PATH, ALIASES_PATH, GPKG_PATH, CROSSWALK_PATH, GEO_REPORT_PATH]
for path in required_paths:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path.relative_to(ROOT)}. Complete the existing audit/geography pipeline first.")

print("Verifying the original GTFS archive...", flush=True)
gtfs_audit = json.loads(GTFS_AUDIT.read_text(encoding="utf-8"))
require(gtfs_audit["source"]["sha256"] == EXPECTED_GTFS_SHA256 == sha256_file(ZIP_PATH), "The GTFS file differs from the audited snapshot; re-audit rather than bypass this check.")
require(gtfs_audit["audit_scope"]["detailed_feed_ids"] == list(SELECTED_FEEDS), "Unexpected detailed-feed audit scope.")
require(gtfs_audit["source_unchanged"] is True and gtfs_audit["helper_self_check_groups"] == 5, "The GTFS audit checkpoint is incomplete.")
require(gtfs_audit["proposed_service_date"] == "2026-10-13", "Proposed date differs from this audited project stage.")
require(gtfs_audit["agency_timezone"] == "Australia/Melbourne", "Unexpected feed timezone.")
geo_spec = load_geo_snapshot(GEO_SPEC_PATH)
geo_report = json.loads(GEO_REPORT_PATH.read_text(encoding="utf-8"))
require(geo_report["boundary_snapshot"] == geo_spec, "The geography report and reviewed configuration differ.")
require(geo_report.get("check_only") is False, "Run the completed geography build, not just --check-only.")
geo_hashes = {key.replace("\\", "/"): value for key, value in geo_report["output_sha256"].items()}
for path in [GPKG_PATH, CROSSWALK_PATH]:
    require(geo_hashes.get(path.relative_to(ROOT).as_posix()) == sha256_file(path), f"Geography output differs from its build report: {path.name}")
for flag in ["gpkg_readback_matches", "geojson_readback_matches", "csv_readback_matches"]:
    require(geo_report.get(flag) is True, f"Geography verification not confirmed: {flag}")

# Check the supplied ABS source as well; we do not re-extract its shapes here.
abs_zip = ROOT / "data/raw/geography" / geo_spec["filename"]
require(abs_zip.is_file() and sha256_file(abs_zip) == geo_spec["sha256"], "ABS boundary ZIP no longer matches the review.")
tracked_paths = required_paths + [abs_zip]
for rel in ["config/rent_snapshot.json", "docs/data_registry.csv", "data/processed/rent_quarterly.csv", "data/processed/rent_quarterly.parquet", "data/processed/rent_quality_report.json", "data/interim/geography/rent_map_2025q3.html"]:
    path = ROOT / rel
    if path.is_file():
        tracked_paths.append(path)
input_hashes = {path: sha256_file(path) for path in tracked_paths}
print("INPUT CHECKPOINT PASSED")
print("GTFS snapshot verified: True")
print("Geography exports match their pipeline report: True")
print("Later service-date proposal:", gtfs_audit["proposed_service_date"], "(not applied to this location inventory)")

Verifying the original GTFS archive...
INPUT CHECKPOINT PASSED
GTFS snapshot verified: True
Geography exports match their pipeline report: True
Later service-date proposal: 2026-10-13 (not applied to this location inventory)


## 3. Load the unchanged 31-LGA layer and reviewed name crosswalk

We reuse the GeoPackage rather than rebuilding the rental/map notebooks. The crosswalk retains both source names and official ABS names. Its three reviewed aliases remain explicit; no stop names are used for spatial matching.

In [3]:
CODE, NAME = "lga_code_2025", "abs_lga_name"
native = gpd.read_file(GPKG_PATH, layer="metro_lga_2025", engine="pyogrio")
require(len(native) == 31 and native.crs is not None and native.crs.to_epsg() == 7844, "Unexpected native LGA layer.")
for column in [CODE, NAME]:
    native[column] = native[column].astype("string")
require(native[CODE].is_unique and native[CODE].str.fullmatch(r"2[0-9]{4}").all(), "Invalid Victorian boundary identifiers.")
require(native.geometry.notna().all() and not native.geometry.is_empty.any() and native.geometry.is_valid.all(), "Invalid or missing LGA shapes.")
require(native.geom_type.isin(["Polygon", "MultiPolygon"]).all(), "Non-area feature.")
crosswalk = pd.read_csv(CROSSWALK_PATH, dtype="string", keep_default_na=False)
require(crosswalk.columns.tolist() == CROSSWALK_COLUMNS and len(crosswalk) == 31, "Unexpected crosswalk.")
require(crosswalk[CODE].is_unique and crosswalk.rental_area_name.is_unique, "Duplicate crosswalk association.")
require(set(crosswalk[CODE]) == set(native[CODE]), "Boundary and rental area sets differ.")
require(set(zip(crosswalk[CODE], crosswalk[NAME])) == set(zip(native[CODE], native[NAME])), "Wrong code/name pair.")
rebuilt_crosswalk = build_crosswalk(crosswalk[["rental_area_name", "source_region"]], native[[CODE, NAME]], load_aliases(ALIASES_PATH))
pd.testing.assert_frame_equal(crosswalk.sort_values("rental_area_name").reset_index(drop=True),
    rebuilt_crosswalk.astype("string").sort_values("rental_area_name").reset_index(drop=True))
require(crosswalk.match_method.value_counts().to_dict() == {"exact_name": 28, "reviewed_alias": 3}, "Reviewed name inventory changed.")
print("Validated LGA features:", len(native), "| Native CRS:", native.crs)
display(crosswalk[["rental_area_name", NAME, CODE]].head())

Validated LGA features: 31 | Native CRS: EPSG:7844


,rental_area_name,abs_lga_name,lga_code_2025
0,Banyule,Banyule,20660
1,Bayside,Bayside (Vic.),20910
2,Boroondara,Boroondara,21110
3,Brimbank,Brimbank,21180
4,Cardinia,Cardinia,21450


## 4. Read only the location tables

The nested archives are copied one at a time into an automatically cleaned temporary buffer; large buffers spill to temporary disk. No archive is extracted into your project. This step reads **only stops.txt** in feeds 2, 3 and 4. It does not rescan stop_times.txt.

All original fields are read as text, including IDs, original coordinate strings and raw wheelchair values. No numeric-ID conversion or cross-feed deduplication occurs.

In [4]:
stop_tables = {}
source_schemas = {}
with ZipFile(ZIP_PATH) as outer:
    require(len(outer.namelist()) == len(set(outer.namelist())), "Duplicate archive member names.")
    for fid in SELECTED_FEEDS:
        member = f"{fid}/google_transit.zip"
        print(f"Reading {member} → stops.txt", flush=True)
        require(member in outer.namelist(), f"Missing audited feed {fid}.")
        with SpooledTemporaryFile(max_size=16 * 1024 * 1024, mode="w+b") as temporary:
            with outer.open(member) as source:
                shutil.copyfileobj(source, temporary, length=1024 * 1024)
            temporary.seek(0)
            with ZipFile(temporary) as inner:
                require(len(inner.namelist()) == len(set(inner.namelist())), "Duplicate inner archive members.")
                with inner.open("stops.txt") as handle:
                    frame = pd.read_csv(handle, dtype="string", keep_default_na=False, encoding="utf-8-sig", low_memory=False)
        require(len(frame) == EXPECTED_LOCATIONS[fid], f"Feed {fid}: location inventory differs from the audit.")
        stop_tables[fid] = frame
        source_schemas[fid] = frame.columns.tolist()
print("All source location records:", sum(map(len, stop_tables.values())))
display(stop_tables["2"][["stop_id", "stop_name", "location_type", "parent_station", "stop_lat", "stop_lon"]].head())

Reading 2/google_transit.zip → stops.txt
Reading 3/google_transit.zip → stops.txt
Reading 4/google_transit.zip → stops.txt
All source location records: 27115


,stop_id,stop_name,location_type,parent_station,stop_lat,stop_lon
0,10117,Jordanville Station,,vic:rail:JOR,-37.87360157,145.11197651
1,10920,Flagstaff Station,,vic:rail:FGS,-37.81205297,144.95562907
2,10921,Flagstaff Station,,vic:rail:FGS,-37.81192588,144.95557578
3,10922,Melbourne Central Station,,vic:rail:MCE,-37.80997342,144.96251307
4,10923,Melbourne Central Station,,vic:rail:MCE,-37.80986514,144.96250469


## 5. Define the preparation and spatial rules

**Location rule:** select `location_type=0` (including blank values, which GTFS defines as 0). Keep station, entrance and internal-node rows in a separate exclusion table. Preserve parent IDs **within their feed**. Raw wheelchair values and parent values are retained without inventing an accessibility score.

**Spatial rule:** consider every polygon intersecting a point. Assign only when there is exactly one candidate and the point is strictly inside it. A shared-edge point can intersect two polygons; an external-edge point can intersect one without being inside. Both need review, not automatic assignment.

| Outcome | Assignment |
|---|---|
| `matched_within` | One unique interior polygon; attach that code |
| `outside_selected_lgas` | No polygon candidate; leave the LGA blank |
| `boundary_review` | Exactly on one candidate's boundary; leave blank |
| `ambiguous_review` | More than one polygon candidate; leave blank |
| `invalid_coordinates` | No usable point; retain the record without an LGA |

Outside means outside **these 31 polygons**, not outside Victoria or necessarily a bad coordinate. We do not use nearest-LGA assignment, snapping, centroid replacement, buffering to force a match, or inner joins that silently drop unmatched source records.

The helper functions are defined below. Start by reading `assign_points()`: it uses a spatial index for candidate discovery and then records all relationships.

In [5]:
# Core functions for notebook 06; copied into a notebook cell during packaging.
STOP_KEYS = ["feed_id", "stop_id"]
STATUS_ORDER = ["matched_within", "outside_selected_lgas", "boundary_review", "ambiguous_review", "invalid_coordinates"]
CODE = "lga_code_2025"
NAME = "abs_lga_name"


def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def validate_areas(areas):
    require(isinstance(areas, gpd.GeoDataFrame) and areas.crs is not None, "LGA polygons need a CRS.")
    require({CODE, NAME, "geometry"}.issubset(areas.columns), "Missing LGA columns.")
    require(areas[CODE].notna().all() and areas[CODE].is_unique, "Duplicate or missing LGA code.")
    require(areas[NAME].notna().all(), "Missing LGA name.")
    require(areas.geometry.notna().all() and not areas.geometry.is_empty.any(), "Missing or empty LGA geometry.")
    require(areas.geometry.is_valid.all(), "Invalid LGA polygon; do not repair silently.")
    require(areas.geom_type.isin(["Polygon", "MultiPolygon"]).all(), "Non-polygon LGA feature.")


def prepare_locations(feed_tables, labels):
    """Preserve every original stops.txt field as text. Add separate derived columns."""
    parts = []
    required = {"stop_id", "stop_name", "stop_lat", "stop_lon", "location_type", "parent_station", "wheelchair_boarding"}
    reserved = {"feed_id", "stop_key", "parent_key", "operational_group", "source_member", "source_record_number", "location_type_effective", "coordinate_valid", "latitude", "longitude", "parent_stop_name", "parent_wheelchair_boarding"}
    for fid, original in feed_tables.items():
        require(required.issubset(original.columns), f"Feed {fid}: expected stops.txt fields are absent.")
        require(not reserved.intersection(original.columns), "Source fields collide with derived fields.")
        frame = original.copy(deep=True)
        require(frame.notna().all().all(), "Use keep_default_na=False when reading original GTFS text.")
        require(frame.stop_id.ne("").all() and frame.stop_id.eq(frame.stop_id.str.strip()).all(), "Blank or padded stop ID.")
        require(frame.stop_id.is_unique, f"Feed {fid}: duplicate stop ID.")
        lt = frame.location_type.replace("", "0")
        require(lt.isin(["0", "1", "2", "3", "4"]).all(), "Invalid location_type.")
        require(frame.parent_station.isin(set(frame.stop_id) | {""}).all(), "Unresolved parent in the same feed.")
        lookup = frame.assign(_location_type=lt).set_index("stop_id")
        children = lt.eq("0") & frame.parent_station.ne("")
        require(frame.loc[children, "parent_station"].map(lookup._location_type).eq("1").all(), "A platform parent is not a station.")
        frame["feed_id"] = fid
        frame["operational_group"] = labels[fid]
        frame["source_member"] = f"{fid}/google_transit.zip!stops.txt"
        frame["source_record_number"] = np.arange(1, len(frame) + 1, dtype=np.int64)
        frame["stop_key"] = fid + ":" + frame.stop_id
        frame["parent_key"] = (fid + ":" + frame.parent_station).where(frame.parent_station.ne(""), "")
        frame["parent_stop_name"] = frame.parent_station.map(lookup.stop_name).fillna("")
        frame["parent_wheelchair_boarding"] = frame.parent_station.map(lookup.wheelchair_boarding).fillna("")
        frame["location_type_effective"] = lt
        frame["latitude"] = pd.to_numeric(frame.stop_lat, errors="coerce").astype("float64")
        frame["longitude"] = pd.to_numeric(frame.stop_lon, errors="coerce").astype("float64")
        frame["coordinate_valid"] = (
            np.isfinite(frame.latitude) & np.isfinite(frame.longitude)
            & frame.latitude.between(-90, 90) & frame.longitude.between(-180, 180)
        )
        parts.append(frame)
    # Optional columns absent in a feed become empty strings. The original per-feed schemas are recorded separately.
    locations = pd.concat(parts, ignore_index=True, sort=False)
    numeric = {"source_record_number", "latitude", "longitude", "coordinate_valid"}
    for col in locations.columns.difference(list(numeric)):
        locations[col] = locations[col].astype("string").fillna("")
    require(not locations.duplicated(STOP_KEYS).any() and locations.stop_key.is_unique, "Combined stop key is not unique.")
    selected = locations.loc[locations.location_type_effective.eq("0")].copy().reset_index(drop=True)
    excluded = locations.loc[~locations.location_type_effective.eq("0")].copy().reset_index(drop=True)
    reasons = {"1": "parent_station_not_platform", "2": "entrance_exit_not_platform", "3": "generic_node_not_platform", "4": "boarding_area_not_platform"}
    excluded["exclusion_reason"] = excluded.location_type_effective.map(reasons).astype("string")
    return locations, selected, excluded


def assign_points(points, areas):
    """Strict interior-only assignment; retain all points and all polygon candidates."""
    validate_areas(areas)
    require(isinstance(points, gpd.GeoDataFrame) and points.crs == areas.crs, "Point and polygon CRS must agree.")
    require(points.stop_key.notna().all() and points.stop_key.is_unique, "Duplicate or missing stop_key.")
    require(not points.duplicated(STOP_KEYS).any(), "Duplicate feed_id/stop_id pair.")
    require(points.loc[points.geometry.notna()].geom_type.eq("Point").all(), "Expected point geometry only.")
    result = points.copy(deep=True).reset_index(drop=True)
    result["_point_row"] = np.arange(len(result), dtype=np.int64)
    valid = result.loc[result.geometry.notna() & ~result.geometry.is_empty & result.geometry.is_valid]
    # An intersecting point may be in an interior or exactly on a boundary.
    joined = gpd.sjoin(valid[["_point_row", "stop_key", "geometry"]],
                       areas[[CODE, NAME, "geometry"]].reset_index(drop=True),
                       how="inner", predicate="intersects")
    if len(joined):
        right_geom = areas.reset_index(drop=True).geometry.iloc[joined.index_right.astype(int)].to_numpy()
        interior = shapely.within(joined.geometry.to_numpy(), right_geom)
        pairs = pd.DataFrame(joined[["_point_row", "stop_key", CODE, NAME]]).copy()
        pairs["relation"] = np.where(interior, "within", "boundary")
        pairs = pairs.sort_values(["stop_key", CODE], kind="stable").reset_index(drop=True)
    else:
        pairs = pd.DataFrame(columns=["_point_row", "stop_key", CODE, NAME, "relation"])
    require(not pairs.duplicated(["stop_key", CODE]).any(), "Duplicate point/polygon candidate.")
    n_candidates = pairs.groupby("_point_row").size()
    n_inside = pairs.loc[pairs.relation.eq("within")].groupby("_point_row").size()
    result["candidate_count"] = result._point_row.map(n_candidates).fillna(0).astype("int64")
    result["interior_count"] = result._point_row.map(n_inside).fillna(0).astype("int64")
    valid_mask = result.geometry.notna() & ~result.geometry.is_empty & result.geometry.is_valid
    status = pd.Series("invalid_coordinates", index=result.index, dtype="string")
    status.loc[valid_mask & result.candidate_count.eq(0)] = "outside_selected_lgas"
    status.loc[valid_mask & result.candidate_count.eq(1) & result.interior_count.eq(0)] = "boundary_review"
    status.loc[valid_mask & result.candidate_count.gt(1)] = "ambiguous_review"
    assigned = valid_mask & result.candidate_count.eq(1) & result.interior_count.eq(1)
    status.loc[assigned] = "matched_within"
    result["spatial_match_status"] = status
    result[CODE] = pd.Series(pd.NA, index=result.index, dtype="string")
    result[NAME] = pd.Series(pd.NA, index=result.index, dtype="string")
    single_rows = result.loc[assigned, "_point_row"]
    single = pairs.loc[pairs._point_row.isin(single_rows)].set_index("_point_row")
    result.loc[assigned, CODE] = result.loc[assigned, "_point_row"].map(single[CODE]).astype("string").to_numpy()
    result.loc[assigned, NAME] = result.loc[assigned, "_point_row"].map(single[NAME]).astype("string").to_numpy()
    codes = pairs.groupby("_point_row")[CODE].agg(lambda x: json.dumps(sorted(x.astype(str)), separators=(",", ":")))
    result["candidate_lga_codes_json"] = result._point_row.map(codes).fillna("[]").astype("string")
    # Audit all candidate associations, not just selected assignments.
    candidate_table = pairs.drop(columns="_point_row").merge(
        pd.DataFrame(result[["stop_key", *STOP_KEYS]]), on="stop_key", how="left", validate="many_to_one")
    candidate_table = candidate_table[["stop_key", *STOP_KEYS, CODE, NAME, "relation"]]
    result = result.drop(columns="_point_row")
    require(len(result) == len(points) and result.stop_key.is_unique, "Spatial join lost or duplicated source rows.")
    require(result[CODE].notna().eq(result.spatial_match_status.eq("matched_within")).all(), "Unreviewed assignment.")
    return result, candidate_table


def mark_near_boundaries(points, areas, radius_m=10.0):
    """Diagnostic flag only: never snap, buffer-assign or change an LGA code."""
    require(points.crs == areas.crs and points.crs.is_projected, "Boundary distance needs the same projected CRS.")
    require(all(a.unit_name == "metre" for a in points.crs.axis_info[:2]), "Projected coordinates must be metres.")
    require(np.isfinite(radius_m) and radius_m > 0, "Positive finite diagnostic radius required.")
    edges = gpd.GeoDataFrame(areas[[CODE]].copy(), geometry=areas.geometry.boundary, crs=areas.crs).reset_index(drop=True)
    valid = points.loc[points.geometry.notna() & ~points.geometry.is_empty]
    near = gpd.sjoin(valid[["stop_key", "geometry"]], edges, how="inner", predicate="dwithin", distance=radius_m)
    return points.stop_key.isin(set(near.stop_key))


def verify_assignments(result, points, areas):
    """Independent per-point checks against all 31 polygons, not another spatial join."""
    require(result.stop_key.tolist() == points.stop_key.tolist(), "Source order/key changed.")
    require(result.spatial_match_status.isin(STATUS_ORDER).all(), "Unknown match status.")
    lookup = areas.set_index(CODE)
    n = 0
    polygons = areas.geometry.to_numpy()
    all_codes = areas[CODE].astype(str).to_numpy()
    for row in result.itertuples(index=False):
        point = row.geometry
        if point is None or point.is_empty or not point.is_valid:
            require(row.spatial_match_status == "invalid_coordinates", "Invalid point received an assignment.")
            continue
        hit = shapely.intersects(polygons, point)
        codes = sorted(all_codes[hit].tolist())
        interiors = int(shapely.contains(polygons, point).sum())
        require(json.loads(row.candidate_lga_codes_json) == codes, "Candidate codes do not match polygon relationships.")
        require(row.candidate_count == len(codes) and row.interior_count == interiors, "Candidate counts disagree.")
        if len(codes) == 1 and interiors == 1:
            require(row.spatial_match_status == "matched_within" and getattr(row, CODE) == codes[0], "Incorrect assigned LGA.")
            require(getattr(row, NAME) == lookup.loc[codes[0], NAME], "Assigned code/name mismatch.")
        elif len(codes) > 1:
            require(row.spatial_match_status == "ambiguous_review" and pd.isna(getattr(row, CODE)), "Ambiguous point forced into an LGA.")
        elif len(codes) == 1:
            require(row.spatial_match_status == "boundary_review" and pd.isna(getattr(row, CODE)), "Boundary point forced into an LGA.")
        else:
            require(row.spatial_match_status == "outside_selected_lgas" and pd.isna(getattr(row, CODE)), "Outside point forced into an LGA.")
        n += 1
    # Geometry copies must remain unchanged throughout the matching routine.
    require(all((a is None and b is None) or (a is not None and b is not None and a.equals_exact(b, 0.0))
                for a, b in zip(result.geometry, points.geometry)), "Point coordinates changed during matching.")
    return n

## 6. Run eight small self-check groups before the real join

These fictional metre-coordinate squares are software fixtures, not Melbourne locations. They test strict interiors, outside points, shared and external boundaries, overlaps, invalid coordinates, duplicate keys, feed namespaces, wrong code assignments, row-order independence and the optional near-boundary screen. They do not modify the real sources or increase the pytest count of 125.

In [6]:
def run_spatial_self_checks():
    from shapely.geometry import Point, Polygon, box
    areas = gpd.GeoDataFrame({CODE: ["A", "B"], NAME: ["Fixture A", "Fixture B"]},
                             geometry=[box(0, 0, 100, 100), box(100, 0, 200, 100)], crs=7855)
    def fixtures(geoms, ids=None, feeds=None):
        n = len(geoms)
        ids = ids or [str(i) for i in range(n)]
        feeds = feeds or ["test"] * n
        return gpd.GeoDataFrame({"feed_id": feeds, "stop_id": ids,
            "stop_key": [a + ":" + b for a, b in zip(feeds, ids)]}, geometry=geoms, crs=7855)
    def rejected(fn):
        try:
            fn()
        except (ValueError, AssertionError):
            return
        raise AssertionError("Deliberately damaged input was accepted.")
    p = fixtures([Point(50, 50), Point(250, 50), Point(0, 50), Point(100, 50), None])
    r, pairs = assign_points(p, areas)
    require(r.spatial_match_status.tolist() == ["matched_within", "outside_selected_lgas", "boundary_review", "ambiguous_review", "invalid_coordinates"], "Classification fixture failed.")
    require(r.loc[1:, CODE].isna().all(), "Non-interior fixture was assigned.")
    require(verify_assignments(r, p, areas) == 4, "Independent fixture verification failed.")
    # 1: known interior/exterior, shared and outer boundaries, missing point.
    overlap = areas.copy(); overlap.loc[1, "geometry"] = box(25, 0, 125, 100)
    o, _ = assign_points(fixtures([Point(50, 50)]), overlap)
    require(o.loc[0, "spatial_match_status"] == "ambiguous_review" and pd.isna(o.loc[0, CODE]), "Overlap was forced.")
    # 2: genuine overlapping interiors.
    same_id = fixtures([Point(50, 50), Point(150, 50)], ids=["same", "same"], feeds=["2", "3"])
    s, _ = assign_points(same_id, areas)
    require(len(s) == 2 and s[CODE].tolist() == ["A", "B"], "Feed namespace lost.")
    rejected(lambda: assign_points(pd.concat([same_id, same_id.iloc[[0]]], ignore_index=True), areas))
    # 3: same ID across feeds preserved; duplicate composite keys rejected.
    duplicate_areas = areas.copy(); duplicate_areas[CODE] = ["A", "A"]
    rejected(lambda: assign_points(p, duplicate_areas))
    invalid_area = areas.copy(); invalid_area.loc[0, "geometry"] = Polygon([(0,0),(100,100),(0,100),(100,0),(0,0)])
    rejected(lambda: assign_points(p, invalid_area))
    # 4: duplicate and invalid polygons.
    tampered = r.copy(); tampered.loc[0, CODE] = "B"; tampered.loc[0, NAME] = "Fixture B"
    rejected(lambda: verify_assignments(tampered, p, areas))
    # 5: wrong code/name assignment independently rejected.
    shuffled = p.sample(frac=1, random_state=19).reset_index(drop=True)
    shuffled_result, _ = assign_points(shuffled, areas)
    cols = ["stop_key", CODE, NAME, "spatial_match_status", "candidate_count", "interior_count", "candidate_lga_codes_json"]
    pd.testing.assert_frame_equal(pd.DataFrame(r[cols]).sort_values("stop_key").reset_index(drop=True),
                                  pd.DataFrame(shuffled_result[cols]).sort_values("stop_key").reset_index(drop=True))
    # 6: changing input row order does not change assignments.
    edge_points = fixtures([Point(5,50), Point(20,50), Point(-5,50), Point(-20,50)])
    require(mark_near_boundaries(edge_points, areas, 10.0).tolist() == [True,False,True,False], "Boundary-distance screen failed.")
    rejected(lambda: mark_near_boundaries(edge_points.to_crs(4326), areas.to_crs(4326)))
    # 7: metre-based proximity flag; geographic degrees rejected.
    raw = pd.DataFrame({"stop_id": ["station", "platform", "bad_coord"], "stop_name": ["S", "P", "X"],
                       "stop_lat": ["-37.8", "-37.8", "95"], "stop_lon": ["145", "145", "145"],
                       "location_type": ["1", "", "0"], "parent_station": ["", "station", ""],
                       "wheelchair_boarding": ["1", "0", "0"]}).astype("string")
    _, selected, excluded = prepare_locations({"2": raw}, {"2": "Fixture feed"})
    require(selected.stop_id.tolist() == ["platform", "bad_coord"] and len(excluded) == 1, "Location hierarchy selection failed.")
    require(selected.parent_key.tolist() == ["2:station", ""], "Namespaced parent not preserved.")
    require(selected.wheelchair_boarding.tolist() == ["0", "0"], "Raw wheelchair values changed.")
    require(selected.coordinate_valid.tolist() == [True, False], "Coordinate range fixture failed.")
    # 8: blank location type = 0; exclude parent stations; retain raw unknowns.
    print("Eight spatial self-check groups passed.")
    return 8

self_check_groups = run_spatial_self_checks()

Eight spatial self-check groups passed.


## 7. Select and account for all location records

One row in the combined table is one **feed/stop identifier**, not one unique physical place. Stop IDs can repeat across feeds. The convenience `stop_key` includes the feed prefix; the original `feed_id` and `stop_id` remain separate.

Keep each platform's own coordinates. Platforms serving the same station can fall on different sides of an LGA boundary; copying the parent station's LGA to all platforms would change the method.

In [7]:
locations, stops, excluded_locations = prepare_locations(stop_tables, FEED_LABELS)
for fid in SELECTED_FEEDS:
    require(int(stops.feed_id.eq(fid).sum()) == EXPECTED_PLATFORMS[fid], f"Unexpected platform selection for feed {fid}.")
require(len(locations) == 27115 and len(stops) == 25031 and len(excluded_locations) == 2084, "Location accounting differs from the audit.")
require(len(stops) + len(excluded_locations) == len(locations), "A source location was lost.")
# Prove the original per-feed fields, values and row order survived the preparation.
for fid, original in stop_tables.items():
    preserved = locations.loc[locations.feed_id.eq(fid), original.columns].reset_index(drop=True)
    pd.testing.assert_frame_equal(preserved, original.reset_index(drop=True), check_exact=True)
require(stops.coordinate_valid.all(), "A reviewed platform now has invalid coordinates. Stop and investigate the changed input.")
location_inventory = pd.DataFrame([
    {"feed_id": fid, "operational_group": FEED_LABELS[fid], "all_location_records": EXPECTED_LOCATIONS[fid],
     "selected_stop_platform_records": int(stops.feed_id.eq(fid).sum()),
     "excluded_non_platform_records": int(excluded_locations.feed_id.eq(fid).sum())}
    for fid in SELECTED_FEEDS
])
shared_raw_stop_ids = int(stops.groupby("stop_id").feed_id.nunique().gt(1).sum())
require(shared_raw_stop_ids == 41, "Cross-feed stop-ID inventory differs from the reviewed source.")
print("Selected stop/platform records:", len(stops))
print("Excluded hierarchy records (retained separately):", len(excluded_locations))
print("Shared raw stop-ID strings across feeds:", shared_raw_stop_ids, "— kept separate")
print("Original stops.txt values preserved: True")
display(location_inventory)

Selected stop/platform records: 25031
Excluded hierarchy records (retained separately): 2084
Shared raw stop-ID strings across feeds: 41 — kept separate
Original stops.txt values preserved: True


,feed_id,operational_group,all_location_records,selected_stop_platform_records,excluded_non_platform_records
0,2,Metropolitan train operational feed,2859,775,2084
1,3,Metropolitan tram,1623,1623,0
2,4,Myki bus: metropolitan + regional town,22633,22633,0


## 8. Create points and use a common coordinate system

GTFS longitude/latitude are WGS84 (`EPSG:4326`). The ABS polygons use GDA2020 (`EPSG:7844`). We **transform copies** of both layers to GDA2020 / MGA zone 55 (`EPSG:7855`) for a common working CRS measured in metres. We do not relabel WGS84 coordinates as GDA2020.

Matching coordinates does not prove survey-level accuracy. WGS84 coordinate realisation/epoch and source positional uncertainty are not specified here. We record the available transformation metadata and later flag near-boundary points; the flag is a sensitivity screen, not a confidence interval.

In [8]:
source_points = gpd.GeoDataFrame(stops.copy(), geometry=gpd.points_from_xy(stops.longitude, stops.latitude), crs=POINT_EPSG)
working_points = source_points.to_crs(MATCH_EPSG)
working_areas = native[[CODE, NAME, "geometry"]].to_crs(MATCH_EPSG).reset_index(drop=True)
validate_areas(working_areas)
require(working_points.geometry.is_valid.all() and not working_points.geometry.is_empty.any(), "Invalid transformed point.")
transformer = pyproj.Transformer.from_crs(POINT_EPSG, MATCH_EPSG, always_xy=True)
transformer.transform(float(stops.longitude.iloc[0]), float(stops.latitude.iloc[0]), errcheck=True)
try:
    used_operation = transformer.get_last_used_operation()
except (AttributeError, pyproj.exceptions.ProjError):
    used_operation = transformer
crs_record = {"source_points": "EPSG:4326", "source_polygons": "EPSG:7844", "working": "EPSG:7855",
              "operation_description": used_operation.description, "operation_definition": used_operation.definition,
              "reported_operation_accuracy_m": used_operation.accuracy,
              "caution": "Operation metadata does not quantify source-coordinate or boundary accuracy. No coordinate epoch was supplied."}
print("Stop source CRS:", source_points.crs, "| Boundary source CRS:", native.crs)
print("Common working CRS:", working_points.crs)
print("No source coordinates or native boundary files were overwritten.")

Stop source CRS: EPSG:4326 | Boundary source CRS: EPSG:7844
Common working CRS: EPSG:7855
No source coordinates or native boundary files were overwritten.


## 9. Match every point and independently verify the result

The first pass uses `gpd.sjoin(..., predicate="intersects")`. We classify its candidates into interiors and exact boundary relationships. The verification then checks each point directly against **all 31 polygons**, without relying on the spatial join's lookup results.

A source table is retained at one row per stop. The separate candidate table can contain multiple rows for an ambiguous stop; this distinction prevents accidental duplication in future service-count joins.

In [9]:
print("Finding all point/polygon candidates...", flush=True)
matched_working, spatial_candidates = assign_points(working_points, working_areas)
print("Independently checking the polygon relationships...", flush=True)
independent_checks = verify_assignments(matched_working, working_points, working_areas)
require(independent_checks == len(stops), "Some valid locations were not independently checked.")
require(matched_working[STOP_KEYS].equals(stops[STOP_KEYS]), "Composite source keys changed.")
print("Independent point checks:", independent_checks)
print("Strict interior assignments:", int(matched_working.spatial_match_status.eq("matched_within").sum()))
print("Outside selected LGAs:", int(matched_working.spatial_match_status.eq("outside_selected_lgas").sum()))
print("Boundary/ambiguous review:", int(matched_working.spatial_match_status.isin(["boundary_review", "ambiguous_review"]).sum()))

Finding all point/polygon candidates...
Independently checking the polygon relationships...
Independent point checks: 25031
Strict interior assignments: 21426
Outside selected LGAs: 3605
Boundary/ambiguous review: 0


## 10. Flag locations close to boundaries — without moving them

The **10-metre threshold is a project-chosen diagnostic radius**, not an official precision standard, walking distance or uncertainty bound. It screens points on either side of any selected LGA boundary for later attention. It never changes `spatial_match_status` or an assigned LGA.

This is distinct from an exact boundary match: a point can be strictly inside one polygon yet close enough to its edge that positional uncertainty might matter. Not every nearby point is wrong, and points farther away are not guaranteed correct.

In [10]:
print("Screening locations within 10 metres of an LGA boundary...", flush=True)
matched_working["near_boundary_10m"] = mark_near_boundaries(working_points, working_areas, BOUNDARY_REVIEW_METRES).to_numpy()
status_index = pd.MultiIndex.from_product([SELECTED_FEEDS, STATUS_ORDER], names=["feed_id", "spatial_match_status"])
status_summary = (matched_working.groupby(["feed_id", "spatial_match_status"]).size()
                  .reindex(status_index, fill_value=0).rename("stop_records").reset_index())
status_summary["operational_group"] = status_summary.feed_id.map(FEED_LABELS)
require(status_summary.stop_records.sum() == len(stops), "Status totals do not account for every stop.")
print("Near-boundary diagnostic records:", int(matched_working.near_boundary_10m.sum()))
display(status_summary.loc[status_summary.stop_records.gt(0), ["feed_id", "operational_group", "spatial_match_status", "stop_records"]])
# The examples demonstrate why each platform keeps its own geographic position.
examples = matched_working.loc[matched_working.feed_id.eq("2") & matched_working.stop_id.isin(["11242", "11244"])]
if len(examples):
    print("Platform example — names may be shared while coordinates fall in different LGAs:")
    display(examples[["feed_id", "stop_id", "stop_name", "parent_key", NAME, "near_boundary_10m"]])

Screening locations within 10 metres of an LGA boundary...
Near-boundary diagnostic records: 443


,feed_id,operational_group,spatial_match_status,stop_records
0,2,Metropolitan train operational feed,matched_within,775
5,3,Metropolitan tram,matched_within,1623
10,4,Myki bus: metropolitan + regional town,matched_within,19028
11,4,Myki bus: metropolitan + regional town,outside_selected_lgas,3605


Platform example — names may be shared while coordinates fall in different LGAs:


,feed_id,stop_id,stop_name,parent_key,abs_lga_name,near_boundary_10m
55,2,11242,Cheltenham Station,2:vic:rail:CTM,Bayside (Vic.),True
57,2,11244,Cheltenham Station,2:vic:rail:CTM,Kingston (Vic.),True


## 11. Prepare the table and reconcile counts

The saved point layer uses the **original WGS84 point coordinates**, not an inverse transform of the working coordinates. All source text fields and raw IDs are preserved. Blank LGA codes are intentional for unassigned statuses.

The 93-row LGA/feed inventory includes all **31 × 3** combinations, including zeros. These are stored location-record counts only. They do not establish active service on 13 October, vehicle mode, frequency, walking access, or affordability. Feed 2 replacement services remain distinguishable later at trip/route level; we do not guess a stop's mode from its folder or its name.

In [14]:
new_columns = [CODE, NAME, "candidate_count", "interior_count", "candidate_lga_codes_json", "spatial_match_status", "near_boundary_10m"]
for col in new_columns:
    require(col not in stops.columns, "Derived column would overwrite a source field.")
attributes = pd.DataFrame(matched_working.drop(columns="geometry")).copy()
attributes["boundary_year"] = 2025
attributes["gtfs_source_sha256"] = EXPECTED_GTFS_SHA256
attributes["stop_coordinate_crs"] = "EPSG:4326"
attributes["spatial_match_crs"] = "EPSG:7855"
attributes[CODE] = attributes[CODE].astype("string").fillna("")
attributes[NAME] = attributes[NAME].astype("string").fillna("")
for col in attributes.columns:
    if col not in {"source_record_number", "latitude", "longitude", "coordinate_valid", "candidate_count", "interior_count", "near_boundary_10m", "boundary_year"}:
        attributes[col] = attributes[col].astype("string").fillna("")
# Original selected rows and values remain exactly the same before output.
pd.testing.assert_frame_equal(attributes[stops.columns], stops, check_exact=True)
matched_points = gpd.GeoDataFrame(attributes, geometry=source_points.geometry.copy(), crs=POINT_EPSG)
require(not attributes.duplicated(STOP_KEYS).any() and attributes.stop_key.is_unique, "Duplicate final key.")
require(attributes[CODE].ne("").eq(attributes.spatial_match_status.eq("matched_within")).all(), "An unreviewed location received a code.")
unassigned = attributes.loc[~attributes.spatial_match_status.eq("matched_within")].copy()
near_boundary_review = attributes.loc[attributes.near_boundary_10m].copy()
assigned_counts = attributes.loc[attributes.spatial_match_status.eq("matched_within")].groupby([CODE, "feed_id"]).size()
all_combinations = pd.MultiIndex.from_product([sorted(native[CODE]), SELECTED_FEEDS], names=[CODE, "feed_id"])
lga_inventory = assigned_counts.reindex(all_combinations, fill_value=0).rename("stored_stop_platform_records").reset_index()
lga_inventory = lga_inventory.merge(crosswalk[[CODE, NAME, "rental_area_name"]], on=CODE, how="left", validate="many_to_one")
lga_inventory["operational_group"] = lga_inventory.feed_id.map(FEED_LABELS)
require(len(lga_inventory) == 93 and lga_inventory.stored_stop_platform_records.sum() == len(attributes) - len(unassigned), "LGA/feed totals do not reconcile.")
print("Final stop rows:", len(attributes), "| Unique feed/stop keys:", attributes.stop_key.nunique())
print("LGA/feed combinations:", len(lga_inventory))
print("Unassigned rows retained:", len(unassigned))
print("Service-date filtering applied: False")
display(lga_inventory.head(9))
print("Outside/review example records (not errors to delete):")
display(unassigned[["feed_id", "stop_id", "stop_name", "spatial_match_status"]].head(8))

Final stop rows: 25031 | Unique feed/stop keys: 25031
LGA/feed combinations: 93
Unassigned rows retained: 3605
Service-date filtering applied: False


,lga_code_2025,feed_id,stored_stop_platform_records,abs_lga_name,rental_area_name,operational_group
0,20660,2,28,Banyule,Banyule,Metropolitan train operational feed
1,20660,3,7,Banyule,Banyule,Metropolitan tram
2,20660,4,554,Banyule,Banyule,Myki bus: metropolitan + regional town
3,20910,2,24,Bayside (Vic.),Bayside,Metropolitan train operational feed
4,20910,3,10,Bayside (Vic.),Bayside,Metropolitan tram
5,20910,4,457,Bayside (Vic.),Bayside,Myki bus: metropolitan + regional town
6,21110,2,45,Boroondara,Boroondara,Metropolitan train operational feed
7,21110,3,241,Boroondara,Boroondara,Metropolitan tram
8,21110,4,433,Boroondara,Boroondara,Myki bus: metropolitan + regional town


Outside/review example records (not errors to delete):


,feed_id,stop_id,stop_name,spatial_match_status
5663,4,13761,Bourke St/Sydney St,outside_selected_lgas
5664,4,13762,Hudson Park/Sydney St,outside_selected_lgas
5834,4,14005,Bloomfield Rd/Queen St,outside_selected_lgas
7422,4,16079,Princes Fwy,outside_selected_lgas
7435,4,16091,Princes Fwy,outside_selected_lgas
7438,4,16096,Yarragon Station/Princes Hwy,outside_selected_lgas
7457,4,16133,Yarragon Railway Station/Princes Hwy,outside_selected_lgas
7522,4,16287,Morwell Station/Princes Dr,outside_selected_lgas


## 12. Save the new outputs, reload, and check

Outputs go **only** under `data/interim/gtfs_spatial/` and to the generated `docs/gtfs_stop_spatial_join.md`. The registry, earlier audits, rental data and boundary inputs are not edited. The CSV is the lookup for later trip/stop-time joins; the GeoPackage contains the same attributes plus point geometry.

Files are staged on the project drive and verified before replacement. Rerunning this final stage replaces only this notebook's own derived outputs. Close these generated files in Excel/QGIS and avoid simultaneous writers before rerunning. No raw data is added to Git.

In [12]:
INTEGER_COLUMNS = ["source_record_number", "candidate_count", "interior_count", "boundary_year"]
FLOAT_COLUMNS = ["latitude", "longitude"]
BOOLEAN_COLUMNS = ["coordinate_valid", "near_boundary_10m"]
TEXT_COLUMNS = [col for col in attributes.columns if col not in INTEGER_COLUMNS + FLOAT_COLUMNS + BOOLEAN_COLUMNS]

def restore_attribute_types(frame):
    require(set(attributes.columns) == set(frame.columns), "Attribute columns changed on read-back.")
    frame = frame[attributes.columns].copy()
    for col in TEXT_COLUMNS:
        frame[col] = frame[col].astype("string").fillna("")
    for col in INTEGER_COLUMNS:
        frame[col] = pd.to_numeric(frame[col], errors="raise").astype("int64")
    for col in FLOAT_COLUMNS:
        frame[col] = pd.to_numeric(frame[col], errors="raise").astype("float64")
    for col in BOOLEAN_COLUMNS:
        text = frame[col].astype("string").str.lower()
        require(text.isin(["true", "false", "1", "0"]).all(), f"Unexpected boolean values in {col}.")
        frame[col] = text.isin(["true", "1"])
    return frame

outputs = {
    "stop_lga_matches.csv": attributes,
    "location_selection_counts.csv": location_inventory,
    "excluded_location_records.csv": excluded_locations,
    "spatial_candidates.csv": spatial_candidates,
    "spatial_match_summary.csv": status_summary,
    "unassigned_stops.csv": unassigned,
    "near_boundary_review.csv": near_boundary_review,
    "stop_records_by_lga_and_feed.csv": lga_inventory,
}
for path, digest in input_hashes.items():
    require(sha256_file(path) == digest, f"Upstream input changed: {path.name}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(prefix="stops-spatial-", dir=OUT_DIR) as folder:
    stage = Path(folder)
    for name, table in outputs.items():
        table.to_csv(stage / name, index=False, encoding="utf-8", lineterminator="\n")
        # Compare CSV values to an in-memory serialisation with the same parser and no NA coercion.
        text_back = pd.read_csv(stage / name, dtype="string", keep_default_na=False)
        text_expected = pd.read_csv(io.StringIO(table.to_csv(index=False)), dtype="string", keep_default_na=False)
        pd.testing.assert_frame_equal(text_back, text_expected, check_exact=True)
    csv_back = pd.read_csv(stage / "stop_lga_matches.csv", dtype={c: "string" for c in TEXT_COLUMNS + BOOLEAN_COLUMNS},
                           keep_default_na=False, float_precision="round_trip")
    csv_back = restore_attribute_types(csv_back)
    pd.testing.assert_frame_equal(csv_back, attributes, check_exact=True)
    gpkg_name = "stop_lga_matches.gpkg"
    pyogrio.write_dataframe(matched_points, stage / gpkg_name, driver="GPKG", layer="stop_lga_matches", geometry_type="Point")
    gpkg_back = gpd.read_file(stage / gpkg_name, layer="stop_lga_matches", engine="pyogrio")
    require(gpkg_back.crs.to_epsg() == POINT_EPSG and len(gpkg_back) == len(attributes), "GeoPackage CRS or row count changed.")
    gpkg_back = gpkg_back.set_index("stop_key").loc[attributes.stop_key].reset_index()
    pd.testing.assert_frame_equal(restore_attribute_types(pd.DataFrame(gpkg_back.drop(columns="geometry"))), attributes, check_exact=True)
    require(all(a.equals_exact(b, 0.0) for a, b in zip(gpkg_back.geometry, source_points.geometry)), "GeoPackage changed original point coordinates.")
    for path, digest in input_hashes.items():
        require(sha256_file(path) == digest, f"Upstream input changed during export: {path.name}")
    summary = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "notebook": "notebooks/06_gtfs_stops_to_lgas.ipynb",
        "source": {"gtfs_filename": "gtfs.zip", "gtfs_sha256": EXPECTED_GTFS_SHA256,
                   "boundary_sha256": geo_spec["sha256"], "boundary_year": 2025,
                   "selected_feeds": list(SELECTED_FEEDS), "source_stops_columns_by_feed": source_schemas},
        "scope": {"lga_definition": "31 LGAs from the rental source's reviewed metropolitan groups",
                  "record_unit": "one (feed_id, stop_id), not one deduplicated physical facility",
                  "location_type": "0 including blank", "service_date_filter_applied": False,
                  "proposed_later_service_date": gtfs_audit["proposed_service_date"],
                  "agency_timezone": gtfs_audit["agency_timezone"], "replacement_trip_filter_applied": False,
                  "no_scores_or_frequencies": True},
        "crs": crs_record,
        "matching_rule": "One candidate polygon and strictly inside it; boundaries and multiple candidates remain unassigned",
        "near_boundary_rule": {"distance_m": BOUNDARY_REVIEW_METRES, "purpose": "diagnostic only; no reassignment or accuracy guarantee"},
        "quality": {"source_location_records": len(locations), "selected_stop_platform_records": len(attributes),
                    "excluded_hierarchy_records": len(excluded_locations), "duplicate_composite_keys": int(attributes.duplicated(STOP_KEYS).sum()),
                    "independent_point_checks": independent_checks, "self_check_groups": self_check_groups,
                    "near_boundary_records": len(near_boundary_review), "unassigned_records": len(unassigned),
                    "source_fields_preserved": True, "csv_readback_matches": True, "gpkg_readback_matches": True,
                    "input_files_unchanged": True},
        "status_counts": status_summary.to_dict("records"),
        "input_sha256": {p.relative_to(ROOT).as_posix(): h for p, h in input_hashes.items()},
        "output_sha256": {name: sha256_file(stage / name) for name in [*outputs, gpkg_name]},
        "versions": {"python": sys.version.split()[0], "pandas": pd.__version__, "geopandas": gpd.__version__,
                     "shapely": shapely.__version__, "pyproj": pyproj.__version__, "proj": pyproj.proj_version_str},
    }
    (stage / "spatial_join_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    # Per-file replacements, not an all-or-nothing multi-file transaction. Manifest is replaced last.
    for name in [*outputs, gpkg_name, "spatial_join_summary.json"]:
        (stage / name).replace(OUT_DIR / name)
print("CSV read-back: matched")
print("GeoPackage attributes and original point coordinates: matched")
print("Saved:", OUT_DIR.relative_to(ROOT))

c:\Users\PhamDang\anaconda3\envs\homereach\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Cannot find tms_NZTM2000.json (GDAL_DATA is not defined)
  ogr_write(


CSV read-back: matched
GeoPackage attributes and original point coordinates: matched
Saved: data\interim\gtfs_spatial


## 13. Record the results and print the final checkpoint

The report is generated; keep personal reflections in your progress file rather than editing generated text that will be replaced next run. No model, date-specific boarding-frequency result or new recommendation has been produced.

In [13]:
report_lines = [
    "# HomeReach Melbourne — Transport-stop spatial join", "",
    f"Generated (UTC): {summary['created_at_utc']}", "",
    "## Inputs and scope", "",
    f"- GTFS snapshot SHA-256: `{EXPECTED_GTFS_SHA256}`.",
    f"- Boundary snapshot: ABS LGA 2025; SHA-256 `{geo_spec['sha256']}`.",
    "- Study area: the same 31 reviewed rental-source metropolitan LGAs, not a newly substituted metro definition.",
    "- Detailed operational feeds: 2, 3 and 4; other five feeds remain outside this stage.",
    f"- All selected-feed location rows: {len(locations):,}; selected stop/platform rows: {len(attributes):,}; hierarchy rows retained separately: {len(excluded_locations):,}.",
    "- GTFS blank location_type is interpreted as 0. Raw values and all source fields are preserved.",
    "- One record is a (feed_id, stop_id) pair. Cross-feed identical IDs and co-located records are not deduplicated.",
    "- Planned later service date: 2026-10-13. NO service-date, active-trip, pickup-policy or replacement-trip filtering is performed here.", "",
    "## Spatial policy", "",
    "- GTFS WGS84 points and ABS GDA2020 polygons are transformed as copies into EPSG:7855 (metres).",
    "- Assign an LGA only for one unique strict-interior polygon; no nearest-LGA, fuzzy-name, snapping or centroid substitution.",
    "- Exact boundaries, overlapping candidates and invalid coordinates remain separate review outcomes.",
    "- Outside the 31 selected polygons is not synonymous with invalid, regional, inactive or inaccessible.",
    "- The 10 m boundary flag is a chosen sensitivity screen only, not an official accuracy threshold or confidence interval.",
    "- Raw coordinate precision, WGS84 epoch/realisation and ABS statistical-boundary uncertainty prevent survey-level certainty.", "",
    "## Observed outcomes", "",
    "| Feed | Outcome | Stop/platform records |", "|---|---|---:|",
]
for row in status_summary.loc[status_summary.stop_records.gt(0)].itertuples(index=False):
    report_lines.append(f"| {row.feed_id} | {row.spatial_match_status} | {row.stop_records:,} |")
report_lines += ["", f"- Independent point/polygon checks: {independent_checks:,}.",
    f"- Near-boundary diagnostic records: {len(near_boundary_review):,}; codes were not changed by this flag.",
    f"- Spatial self-check groups: {self_check_groups}; these are notebook checks, not new pytest tests.",
    "- CSV values and GeoPackage attributes/point coordinates matched on read-back.",
    "- Original GTFS, native polygons, reviewed crosswalk, rental files, earlier audits and registry were not overwritten.", "",
    "## Interpretation", "",
    "Stored location-record counts are not unique physical stops, railway stations, daily services, frequencies, walking accessibility, or transport quality.",
    "Platforms with the same station name may legitimately lie in different LGAs. Do not force them to inherit a parent station's spatial assignment.",
    "Feed 2 includes replacement-bus operations; infer service mode/flags later from reviewed route/trip information rather than just the feed number or a stop name.",
    "Unknown wheelchair codes and parent values remain raw; no accessible-journey claim is made.",
    "A stop outside an LGA can still serve its residents. Point-in-polygon membership is not a residential catchment analysis.",
    "Rent describes 2025Q3; the proposed transport date is in October 2026. Never insert this newer snapshot into historical rental-model evaluation as past transport evidence.", "",
    "## Outputs", "",
    "`data/interim/gtfs_spatial/`: eight CSV tables, `stop_lga_matches.gpkg`, and `spatial_join_summary.json`.",
    "The complete assignment CSV preserves all selected records, including unassigned ones. Candidate associations are a separate table.",
    "All 31 × 3 LGA/feed combinations appear in the location inventory; zero means no assigned stored record, not proven absence of usable services.", "",
    "## Next stage", "",
    "Join active trips and boardable stop-time events using both feed and stop/trip IDs; apply calendar exceptions, pickup policies, onward-stop rules and replacement-service flags before calculating service indicators.", "",
    "## References", "",
    "- https://gtfs.org/documentation/schedule/reference/", 
    "- https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html",
    "- https://geopandas.org/en/stable/docs/user_guide/projections.html",
    "- https://opendata.transport.vic.gov.au/dataset/gtfs-schedule",
]
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
for path, digest in input_hashes.items():
    require(sha256_file(path) == digest, f"Upstream input changed: {path.name}")
print("STOP SPATIAL JOIN CHECKPOINT PASSED")
print("All source location records:", len(locations))
print("Selected stop/platform records:", len(attributes))
print("Excluded hierarchy records retained:", len(excluded_locations))
print("Matched within selected LGAs:", int(attributes.spatial_match_status.eq('matched_within').sum()))
print("Outside selected LGAs retained:", int(attributes.spatial_match_status.eq('outside_selected_lgas').sum()))
print("Exact-boundary review:", int(attributes.spatial_match_status.eq('boundary_review').sum()))
print("Ambiguous review:", int(attributes.spatial_match_status.eq('ambiguous_review').sum()))
print("Invalid coordinates:", int(attributes.spatial_match_status.eq('invalid_coordinates').sum()))
print("Duplicate feed/stop keys:", int(attributes.duplicated(STOP_KEYS).sum()))
print("Independent point checks:", independent_checks)
print("Near-boundary diagnostic records:", len(near_boundary_review))
print("Spatial self-check groups:", self_check_groups, "passed")
print("LGA/feed inventory rows:", len(lga_inventory))
print("CSV and GeoPackage read-back: matched")
print("Service-date filter applied: False")
print("Upstream inputs unchanged: True")
print("Saved report:", REPORT_PATH.relative_to(ROOT))
print("Saved outputs:", OUT_DIR.relative_to(ROOT))

STOP SPATIAL JOIN CHECKPOINT PASSED
All source location records: 27115
Selected stop/platform records: 25031
Excluded hierarchy records retained: 2084
Matched within selected LGAs: 21426
Outside selected LGAs retained: 3605
Exact-boundary review: 0
Ambiguous review: 0
Invalid coordinates: 0
Duplicate feed/stop keys: 0
Independent point checks: 25031
Near-boundary diagnostic records: 443
Spatial self-check groups: 8 passed
LGA/feed inventory rows: 93
CSV and GeoPackage read-back: matched
Service-date filter applied: False
Upstream inputs unchanged: True
Saved report: docs\gtfs_stop_spatial_join.md
Saved outputs: data\interim\gtfs_spatial


## Checkpoint before continuing

Save the notebook with **Ctrl+S**. Send the final checkpoint or the first error, not screenshots of every cell. Do not delete outside records to make the match rate 100%. The Myki feed contains locations outside our 31-LGA study area.

You should be able to explain: **we mapped stored stop/platform coordinates, preserved feed-specific IDs and all unmatched records, and have not yet counted usable scheduled services.**

A few sources are close to administrative boundaries; the 10 m diagnostic flag records that sensitivity without silently altering assignments. Keep this caveat when comparing future LGA totals.